<a href="https://colab.research.google.com/github/wtree101/ZIP-RC-Colab/blob/main/notebooks/colab/04_prompt_split.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 4 — 按 prompt 切分 train / validation / test

同一题的所有 rollout 必须属于同一个 split，避免 trajectory-level leakage。

默认使用 80% / 10% / 10%，并验证三个集合的 prompt 完全不重叠。

先运行 `colab/00_memory_and_config.ipynb`；本 Notebook 的训练命令会自动使用 ZIP mamba 环境。

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO = Path("/content/ZIP-RC-Colab")
ZIP_PY = Path("/content/mamba/envs/zip/bin/python")

if not REPO.exists():
    raise FileNotFoundError("远端仓库不存在；请先运行 colab/00_memory_and_config.ipynb。")
if not ZIP_PY.exists():
    raise FileNotFoundError("ZIP Python 环境不存在；请先运行 colab/00_memory_and_config.ipynb。")

os.environ["ZIPRC_PYTHON"] = str(ZIP_PY)
sys.path.insert(0, str(REPO / "notebooks"))
from ziprc_notebook_utils import *

CONFIG = load_config(REPO)
print("Repository:", REPO)
print("ZIP Python:", ZIP_PY)
print("Experiment:", CONFIG["experiment_name"])

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

source_path = REPO / CONFIG["paths"]["full"]
df = pd.read_parquet(source_path)
prompt_ids = np.array(sorted(df["prompt_idx"].unique()))
rng = np.random.default_rng(42)
rng.shuffle(prompt_ids)

n_prompts = len(prompt_ids)
train_end = int(n_prompts * CONFIG["train_fraction"])
validation_end = train_end + int(n_prompts * CONFIG["validation_fraction"])
assignments = {
    "train": set(prompt_ids[:train_end].tolist()),
    "validation": set(prompt_ids[train_end:validation_end].tolist()),
    "test": set(prompt_ids[validation_end:].tolist()),
}

split_frames = {}
for name, ids in assignments.items():
    split_frame = df[df["prompt_idx"].isin(ids)].copy().reset_index(drop=True)
    split_frame["data_split"] = name
    path = REPO / CONFIG["paths"][name]
    path.parent.mkdir(parents=True, exist_ok=True)
    split_frame.to_parquet(path, index=False)
    split_frames[name] = split_frame
    print(name, path, len(split_frame))

In [ ]:
import matplotlib.pyplot as plt

summary = pd.DataFrame([
    {
        "split": name,
        "prompts": frame["prompt_idx"].nunique(),
        "rollouts": len(frame),
        "accuracy": frame["correct"].mean(),
        "finished_rate": frame["finished"].mean(),
        "median_length": frame["length"].median(),
    }
    for name, frame in split_frames.items()
])
display(summary.round(3))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
summary.set_index("split")[["prompts", "rollouts"]].plot.bar(ax=axes[0])
axes[0].set_title("Split sizes")
summary.set_index("split")["accuracy"].plot.bar(ax=axes[1], color="#49beaa", ylim=(0, 1))
axes[1].set_title("Correctness stability")
for name, frame in split_frames.items():
    axes[2].hist(frame["length"], bins=30, histtype="step", density=True, label=name)
axes[2].set(title="Length distributions", xlabel="tokens")
axes[2].legend()
plt.tight_layout()
plt.show()

overlaps = {
    "train∩validation": len(assignments["train"] & assignments["validation"]),
    "train∩test": len(assignments["train"] & assignments["test"]),
    "validation∩test": len(assignments["validation"] & assignments["test"]),
}
checks = [
    gate("无 prompt 泄漏", sum(overlaps.values()) == 0, str(overlaps)),
    gate("所有数据均已分配", sum(len(frame) for frame in split_frames.values()) == len(df), f"{sum(len(frame) for frame in split_frames.values())}/{len(df)}"),
    gate("每个 split 正负标签都有", all(frame["correct"].nunique() == 2 for frame in split_frames.values()), str({name: frame['correct'].value_counts().to_dict() for name, frame in split_frames.items()}), kind="scientific"),
    gate("Split accuracy 漂移 ≤10pp", summary["accuracy"].max() - summary["accuracy"].min() <= .10, f"range={summary['accuracy'].max() - summary['accuracy'].min():.1%}", kind="scientific"),
]
display(gate_frame(checks))
save_stage_report(REPO, "04_prompt_split", checks, {"summary": summary.to_dict(orient="records"), "overlaps": overlaps})